# Consultando Dados: select(), Filtros e Paginação

---

## Contexto: o time de analytics quer os dados

O nosso banco está já está populado, agora o time de vendas pediu alguns relatórios:
- "Lista os 5 produtos mais caros da categoria Eletrônicos"
- "Quais clientes são de SP ou RJ?"
- "Preciso paginar os pedidos em 20 por página"

Nesta aula você vai:
- Usar o `select()` moderno do SQLAlchemy 2.x
- Escolher o formato de retorno certo para cada situação
- Filtrar com `where()`, `and_()`, `or_()` e `like()`
- Ordenar e paginar resultados
- Usar IA para transformar perguntas em queries

---

## 1. Configuração

In [ ]:
from pathlib import Path
from datetime import datetime
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index
)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

In [ ]:
# código:

## 2. O básico do select()

No SQLAlchemy 2.x, toda consulta começa com `select()`.  
Importante: o `select()` apenas **monta** a instrução, ele não vai ao banco ainda.

```
stmt = select(...)           → monta a query (em memória)
session.execute(stmt)        → executa e retorna linhas
session.scalars(stmt)        → executa e retorna objetos ORM
```

### 2.1 Retornando objetos ORM com `scalars()`

Use quando quiser trabalhar com os **objetos completos** acessar atributos, navegar relacionamentos:

In [ ]:
# código:

### 2.2 Retornando colunas específicas

Use quando só precisar de alguns campos, trafega menos dado e é mais rápido:

In [ ]:
# código:

### 2.3 Retornando dicionários com `mappings()`

Use quando quiser acessar campos pelo nome — **ótimo para passar para o Pandas**:

In [ ]:
# código:

### Resumo: qual método usar?

| O que você quer | Como fazer | Retorna |
|---|---|---|
| Objeto ORM completo | `session.scalars(select(Cliente)).all()` | `[<Cliente>, ...]` |
| Colunas específicas | `session.execute(select(Cliente.id, Cliente.nome)).all()` | `[(1, 'Ana'), ...]` |
| Dicionários | `session.execute(stmt).mappings().all()` | `[{'id': 1, 'nome': 'Ana'}, ...]` |

> ⚠️ **Armadilha:** `scalars()` com múltiplas colunas retorna **apenas a primeira coluna**!  
> Para múltiplas colunas, sempre use `execute().all()`.

---

## 3. Filtrando resultados

### 3.1 Filtro simples com `where()`

In [ ]:
# código:

### 3.2 Condições com OR

In [ ]:
# código:

### 3.3 Busca com LIKE

In [ ]:
# código:

### 3.4 Montando queries dinamicamente

No trabalho, muitas vezes os filtros são opcionais (vêm de um formulário, de uma API...).  
O padrão abaixo é muito comum em código de produção:

In [ ]:
# código:

## 4. Ordenando resultados

In [ ]:
# código:

## 5. Paginação com limit() e offset()

Essencial para APIs e dashboards que não podem carregar tudo de uma vez.

In [ ]:
# código:

---

## Exercício: Usando IA para isso

Uma das melhores aplicações de IA no dia a dia de dados: **transformar perguntas em código de query**.

**Prompt para traduzir perguntas em queries:**
```
Usando os modelos SQLAlchemy abaixo, escreva uma query Python que responda:
"Quais são os 5 produtos mais vendidos do mês passado,
agrupados por categoria, com o total de receita de cada?"

[cole os modelos ORM aqui]
```
---

### Resposta:Código gerado pelo Copilot

In [ ]:
from sqlalchemy import select, func
from datetime import datetime, timedelta

# Calcula o mês passado
hoje = datetime.now()
mes_passado = hoje.replace(day=1) - timedelta(days=1)
inicio_mes_passado = mes_passado.replace(day=1)
fim_mes_passado = mes_passado

with Session(engine) as session:
    # Query para os 5 produtos mais vendidos do mês passado, agrupados por categoria
    stmt = (
        select(
            Produto.categoria,  # Agrupa por categoria
            Produto.nome,  # Nome do produto
            func.sum(ItemPedido.quantidade).label("total_vendido"),  # Soma da quantidade vendida
            func.sum(ItemPedido.preco_venda * ItemPedido.quantidade).label("receita_total")  # Soma da receita (preço * quantidade)
        )
        .join(ItemPedido, ItemPedido.produto_id == Produto.id)  # JOIN com ItemPedido
        .join(Pedido, Pedido.id == ItemPedido.pedido_id)  # JOIN com Pedido
        .where(Pedido.data_pedido >= inicio_mes_passado)  # Filtra pedidos do mês passado
        .where(Pedido.data_pedido <= fim_mes_passado)
        .group_by(Produto.categoria, Produto.nome)  # Agrupa por categoria e nome do produto
        .order_by(func.sum(ItemPedido.quantidade).desc())  # Ordena por total vendido (desc)
        .limit(5)  # Limita aos 5 primeiros
    )

    resultados = session.execute(stmt).all()

    # Exibe os resultados
    print("5 Produtos Mais Vendidos do Mês Passado (por Categoria):")
    print(f"{'Categoria':<20} {'Produto':<30} {'Total Vendido':<15} {'Receita Total':>15}")
    print("-" * 80)
    for row in resultados:
        print(f"{row.categoria:<20} {row.nome:<30} {row.total_vendido:<15} R${row.receita_total:>14.2f}")